# Global Sensitivity C — `sample_weight` Audit

이 Notebook은 공식 Global Stage 1~4 결과를 바꾸지 않는다. 기존 고정 Global **Train** 행에서 Stage 3의 확정 25개 Feature를 유지하고, Train 안의 각 SAMPID 반복관측 수 `n_i`에 대해 `sample_weight = 1 / n_i`만 점검한다. 모델 학습·CV·OOF·Test 접근·예측·metric 계산은 포함하지 않는다.

가중치는 이후 모델 `fit`에만 전달하는 입력이며, validation metric 또는 OOF F1에는 적용하지 않는다.

## 1. 입력 경로와 공통 함수

공식 baseline 저장 산출물, 고정 split, Stage 3 Feature 목록만 지정합니다. 이 단계에서는 데이터를 계산하거나 모델을 학습하지 않습니다.


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path(os.environ.get('KHUDA_PROJECT_ROOT', Path.cwd())).resolve()
while not (ROOT / 'code').is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError('KHUDA_PROJECT_ROOT에 저장소 루트를 지정하거나 저장소 안에서 Notebook을 실행하세요.')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if 'code' in sys.modules and not hasattr(sys.modules['code'], '__path__'):
    del sys.modules['code']

from code.pipeline.audit import (
    calculate_train_sample_weight,
    load_saved_global_train_frame,
    load_selected_feature_names,
)

RESULT_ROOT = ROOT / 'data' / 'result' / 'baseline_42features'
DATASET_PATH = RESULT_ROOT / 'datasets' / 'global_dataset.parquet'
SPLIT_PATH = RESULT_ROOT / 'splits' / 'split_ids.csv'
SELECTED_FEATURES_PATH = RESULT_ROOT / 'modeling' / 'stage_3' / 'selected_features.csv'

required_paths = [DATASET_PATH, SPLIT_PATH, SELECTED_FEATURES_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError('필요한 공식 Global 저장 산출물이 없습니다:' + chr(10) + chr(10).join(missing_paths))

## 2. Global Train 및 Feature 계약

기존 Train SAMPID만 선택하고, Stage 3의 확정 25개 Feature가 그대로 유지되는지 확인합니다.


In [ ]:
# 기존 Train SAMPID만 사용한다. 새 Feature와 Test Dataset은 만들지 않는다.
train_frame = load_saved_global_train_frame(DATASET_PATH, SPLIT_PATH)
selected_features = load_selected_feature_names(SELECTED_FEATURES_PATH)
missing_features = [feature for feature in selected_features if feature not in train_frame.columns]
if missing_features:
    raise ValueError('저장된 Global Train에 Stage 3 선택 Feature가 없습니다: ' + ', '.join(missing_features))
X_C = train_frame.loc[:, selected_features].copy()
train_audit = train_frame.loc[:, ['SAMPID', 'baseline_year', 'target_year', 'employment_transition']].copy()
train_audit['sample_weight'] = calculate_train_sample_weight(train_audit['SAMPID'])

display(pd.DataFrame([{
    'train_rows': len(train_audit),
    'train_unique_SAMPID': train_audit['SAMPID'].nunique(),
    'C_feature_count': X_C.shape[1],
    'n_prior_periods_in_predictors': 'n_prior_periods' in X_C.columns,
    'test_used': False,
}]))
display(pd.DataFrame({'feature_order': range(1, len(selected_features) + 1), 'feature': selected_features}))

## 3. SAMPID별 행 수와 `sample_weight`

Global Train 안에서 SAMPID별 행 수 `n_i`를 기준으로 각 행의 `sample_weight = 1 / n_i`를 계산하고 분포와 사람별 합계를 확인합니다.


In [ ]:
# SAMPID별 행 수와 행 단위 sample_weight 분포
row_count_by_sampid = train_audit.groupby('SAMPID').size().rename('person_period_rows')
display(row_count_by_sampid.value_counts().sort_index().rename('SAMPID_count').to_frame())
display(train_audit['sample_weight'].value_counts(dropna=False).sort_index().rename('row_count').to_frame())
display(train_audit['sample_weight'].agg(['min', 'max', 'mean']).to_frame('sample_weight'))

weight_sum_by_sampid = train_audit.groupby('SAMPID')['sample_weight'].sum().rename('sample_weight_sum')
display(weight_sum_by_sampid.reset_index().sort_values('SAMPID'))

## 4. 기준연도·Target class별 가중치 요약

baseline year별 가중치 분포와 target class별 raw row count·weighted total을 함께 확인합니다.


In [ ]:
# 기준연도별 가중치 분포, 그리고 raw row count와 weighted total의 target class별 비교
baseline_year_weight_distribution = (
    train_audit.groupby('baseline_year')['sample_weight']
    .agg(['count', 'min', 'max', 'mean', 'sum'])
    .reset_index()
)
display(baseline_year_weight_distribution)
target_class_summary = (
    train_audit.groupby('employment_transition')
    .agg(raw_row_count=('employment_transition', 'size'), weighted_total=('sample_weight', 'sum'))
    .reset_index()
)
display(target_class_summary)

## 5. 실행 시 검증할 assertions

아래 assertion은 사람이 Notebook을 실행할 때만 동작합니다. 가중치 범위, 사람별 합계, Feature 계약을 자동 점검합니다.


In [ ]:
# Audit assertions — 이 셀은 사람이 Notebook을 실행할 때만 검사한다.
assert len(selected_features) == 25, 'Stage 3 selected_features.csv는 정확히 25개여야 합니다.'
assert X_C.shape[1] == 25, 'C Feature 수는 정확히 25개여야 합니다.'
assert 'n_prior_periods' not in X_C.columns, 'C predictor에는 n_prior_periods가 포함되면 안 됩니다.'
assert train_audit['sample_weight'].gt(0).all(), 'sample_weight는 0보다 커야 합니다.'
assert train_audit['sample_weight'].le(1).all(), 'sample_weight는 1 이하여야 합니다.'
assert weight_sum_by_sampid.sub(1.0).abs().le(1e-12).all(), '각 SAMPID의 sample_weight 합은 1이어야 합니다.'
print('All C audit assertions passed.')